✅ CELL 1) Imports + 기본 설정 + DB 연결

In [1]:
import time
import json
import pickle
from dataclasses import dataclass
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

# -------------------------
# DB CONFIG (너가 준 값)
# -------------------------
DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

ML_SCHEMA   = "h_machine_learning"
SRC_TABLE   = "1_database_house"
MODEL_TABLE = "3_machine_learning_model"

CURSOR_TABLE = "rt_cursor"
ALARM_TABLE  = "rt_alarms"

CURSOR_KEY = "realtime_Brule_FH1"

# -------------------------
# 운영 파라미터
# -------------------------
POLL_SEC = 10
WINDOW_MINUTES = 60
LOOKBACK_HOURS_ON_BOOT = 6

TOPK_PER_HOUR_PER_STATION = 1
MIN_PROB_LIST = [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

SELECTED_MIN_PROB = 0.6  # 실제 운영 알람 insert에 사용할 min_prob(정책으로 바꿀 것)

def make_engine(cfg: dict) -> Engine:
    url = (
        f"postgresql+psycopg2://{cfg['user']}:{cfg['password']}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

ENGINE = make_engine(DB_CONFIG)
print("[OK] engine ready")

[OK] engine ready


✅ CELL 2) 테이블 컬럼 자동 감지 (step_key 자동 선택)

In [2]:
def table_columns(engine: Engine, schema: str, table: str) -> set[str]:
    q = """
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema=:schema AND table_name=:table
    """
    with engine.connect() as conn:
        rows = conn.execute(text(q), {"schema": schema, "table": table}).fetchall()
    return {r[0] for r in rows}

SRC_COLS = table_columns(ENGINE, ML_SCHEMA, SRC_TABLE)

COL_END_TS  = "end_ts"
COL_STATION = "station"
COL_BARCODE = "barcode_information"

# step 컬럼 자동 선택: step_description -> group -> remark -> 'NA'
if "step_description" in SRC_COLS:
    COL_STEPKEY = "step_description"
elif "group" in SRC_COLS:
    COL_STEPKEY = "group"
elif "remark" in SRC_COLS:
    COL_STEPKEY = "remark"
else:
    COL_STEPKEY = None

print("[SRC_COLS] has:", sorted(list(SRC_COLS))[:20], "...")
print("[STEPKEY] selected:", COL_STEPKEY)

[SRC_COLS] has: ['barcode_information', 'ct_max', 'ct_mean', 'ct_sum', 'end_day', 'end_time', 'end_ts', 'event_type', 'fail_cnt', 'fail_cnt__tag1__equipment', 'fail_cnt__tag2__equipment', 'fail_dev_max', 'fail_dev_sum', 'fail_rate', 'fail_rate__tag1__equipment', 'fail_rate__tag2__equipment', 'group', 'problem1', 'problem2', 'problem3'] ...
[STEPKEY] selected: group


✅ CELL 3) Cursor/Alarm 테이블 생성 (DDL)

In [3]:
DDL_CURSOR = f"""
CREATE TABLE IF NOT EXISTS "{ML_SCHEMA}"."{CURSOR_TABLE}" (
    cursor_key      text PRIMARY KEY,
    cursor_end_ts   timestamp,
    updated_at      timestamp NOT NULL DEFAULT now()
);
"""

DDL_ALARM = f"""
CREATE TABLE IF NOT EXISTS "{ML_SCHEMA}"."{ALARM_TABLE}" (
    end_ts          timestamp NOT NULL,
    station         text      NOT NULL,
    barcode         text      NOT NULL,
    step_key        text      NOT NULL,
    score           float8    NOT NULL,
    min_prob        float8    NOT NULL,
    alarm_level     text      NOT NULL,
    created_at      timestamp NOT NULL DEFAULT now(),
    extra_json      jsonb,
    CONSTRAINT rt_alarms_uniq UNIQUE(end_ts, station, barcode, step_key, min_prob)
);
"""

def ensure_runtime_tables(engine: Engine):
    with engine.begin() as conn:
        conn.execute(text(DDL_CURSOR))
        conn.execute(text(DDL_ALARM))

ensure_runtime_tables(ENGINE)
print("[OK] runtime tables ensured:", CURSOR_TABLE, ALARM_TABLE)

[OK] runtime tables ensured: rt_cursor rt_alarms


✅ CELL 4) Cursor read/write

In [4]:
def read_cursor(engine: Engine, cursor_key: str) -> datetime:
    q = f'SELECT cursor_end_ts FROM "{ML_SCHEMA}"."{CURSOR_TABLE}" WHERE cursor_key=:k'
    with engine.connect() as conn:
        r = conn.execute(text(q), {"k": cursor_key}).fetchone()
    if r and r[0] is not None:
        return pd.Timestamp(r[0]).to_pydatetime()
    return datetime(1970, 1, 1)

def write_cursor(engine: Engine, cursor_key: str, cursor_ts: datetime):
    q = f"""
    INSERT INTO "{ML_SCHEMA}"."{CURSOR_TABLE}"(cursor_key, cursor_end_ts, updated_at)
    VALUES (:k, :ts, now())
    ON CONFLICT (cursor_key)
    DO UPDATE SET cursor_end_ts=EXCLUDED.cursor_end_ts, updated_at=now()
    """
    with engine.begin() as conn:
        conn.execute(text(q), {"k": cursor_key, "ts": cursor_ts})

✅ CELL 5) Active 모델 로드 (is_active=true 최신 1개)

In [5]:
@dataclass
class LoadedModel:
    booster: object
    feature_cols: list[str]
    meta: dict

def load_active_model(engine: Engine, target_key: str = "sparepart") -> LoadedModel:
    q = f"""
    SELECT
        updated_date,
        model_blob,
        feature_cols_json,
        label_config_json,
        policy_json,
        metrics_json,
        train_config_json,
        model_name,
        model_version,
        target_key
    FROM "{ML_SCHEMA}"."{MODEL_TABLE}"
    WHERE is_active = true
      AND (target_key = :tk OR :tk IS NULL)
    ORDER BY updated_date DESC
    LIMIT 1
    """
    with engine.connect() as conn:
        row = conn.execute(text(q), {"tk": target_key}).fetchone()
    if not row:
        raise RuntimeError("active model not found. 3_machine_learning_model에 is_active=true 모델이 있어야 합니다.")

    updated_date, blob, feature_cols_json, label_json, policy_json, metrics_json, train_json, model_name, model_version, tk = row
    booster = pickle.loads(blob)

    # jsonb/text 모두 대응
    if isinstance(feature_cols_json, (dict, list)):
        feature_cols = list(feature_cols_json)
    else:
        feature_cols = json.loads(feature_cols_json)

    meta = {
        "updated_date": str(updated_date),
        "model_name": model_name,
        "model_version": model_version,
        "target_key": tk,
        "label_config": label_json if isinstance(label_json, dict) else json.loads(label_json) if label_json else {},
        "policy_json": policy_json if isinstance(policy_json, dict) else json.loads(policy_json) if policy_json else {},
        "metrics_json": metrics_json if isinstance(metrics_json, dict) else json.loads(metrics_json) if metrics_json else {},
        "train_config_json": train_json if isinstance(train_json, dict) else json.loads(train_json) if train_json else {},
    }
    return LoadedModel(booster=booster, feature_cols=feature_cols, meta=meta)

MODEL = load_active_model(ENGINE, target_key="sparepart")
print("[OK] active model loaded:", MODEL.meta["model_name"], MODEL.meta["model_version"])
print("[OK] feature n =", len(MODEL.feature_cols))
print("[OK] feature head:", MODEL.feature_cols[:12])

[OK] active model loaded: CELL10v3_B_rule_LGBM_FH1 2025-09-27_2025-12-05_v3
[OK] feature n = 9
[OK] feature head: ['ct_mean', 'ct_max', 'ct_sum', 'step_cnt', 'value_mean', 'value_std', 'value_amp', 'value_diff_mean', 'value_diff_std']


✅ CELL 6) DB에서 새 row 읽기 (커서 + WINDOW 하이브리드)

In [6]:
def fetch_rows(engine: Engine, model: LoadedModel, start_ts: datetime, end_ts: datetime, limit: int = 200_000) -> pd.DataFrame:
    step_select = f'"{COL_STEPKEY}" AS step_key' if COL_STEPKEY else "'NA' AS step_key"

    # 실제 테이블에 존재하는 feature만 SELECT
    feat_cols = [c for c in model.feature_cols if c in SRC_COLS]
    feat_select = ", ".join([f'"{c}"' for c in feat_cols]) if feat_cols else ""

    q = f"""
    SELECT
        "{COL_END_TS}"   AS end_ts,
        "{COL_STATION}"  AS station,
        "{COL_BARCODE}"  AS barcode,
        {step_select}
        {"," if feat_select else ""} {feat_select}
    FROM "{ML_SCHEMA}"."{SRC_TABLE}"
    WHERE "{COL_END_TS}" > :start_ts
      AND "{COL_END_TS}" <= :end_ts
    ORDER BY "{COL_END_TS}" ASC, "{COL_STATION}" ASC, "{COL_BARCODE}" ASC
    LIMIT {int(limit)}
    """
    with engine.connect() as conn:
        df = pd.read_sql(text(q), conn, params={"start_ts": start_ts, "end_ts": end_ts})
    return df

✅ CELL 7) 부팅 시 커서 초기화 + lookback 재구성(옵션)

In [7]:
def init_cursor_on_boot(engine: Engine, cursor_key: str, lookback_hours: int = 6) -> datetime:
    cur = read_cursor(engine, cursor_key)
    if cur.year <= 1971:
        cur = datetime.now() - timedelta(hours=lookback_hours)
        write_cursor(engine, cursor_key, cur)
        print("[BOOT] cursor initialized:", cur)
    else:
        print("[BOOT] cursor loaded:", cur)
    return cur

cursor_ts = init_cursor_on_boot(ENGINE, CURSOR_KEY, lookback_hours=LOOKBACK_HOURS_ON_BOOT)

[BOOT] cursor loaded: 2026-01-21 05:32:44.209791


✅ CELL 8) 모델 점수 + 정책(Top1/hour×station + min_prob)

In [8]:
def predict_scores(model: LoadedModel, df: pd.DataFrame) -> np.ndarray:
    X = df.copy()
    for c in model.feature_cols:
        if c not in X.columns:
            X[c] = np.nan
    X = X[model.feature_cols].astype(float)
    s = model.booster.predict(X, num_iteration=getattr(model.booster, "best_iteration", None))
    return np.asarray(s, dtype=float)

def apply_top1_per_hour_station(df_scored: pd.DataFrame, min_prob: float) -> pd.DataFrame:
    if df_scored.empty:
        return df_scored

    df = df_scored.copy()
    df = df[df["score"] >= float(min_prob)]
    if df.empty:
        return df

    df["hour_bucket"] = df["end_ts"].dt.floor("H")
    df = df.sort_values(["hour_bucket", "station", "score"], ascending=[True, True, False])
    df = df.groupby(["hour_bucket", "station"], as_index=False).head(TOPK_PER_HOUR_PER_STATION)
    return df

✅ CELL 9) 알람 insert (중복 방지 UNIQUE)

In [9]:
def insert_alarms(engine: Engine, df_alarm: pd.DataFrame, min_prob: float, alarm_level: str = "MODEL_EARLY") -> int:
    if df_alarm.empty:
        return 0

    q = f"""
    INSERT INTO "{ML_SCHEMA}"."{ALARM_TABLE}"
    (end_ts, station, barcode, step_key, score, min_prob, alarm_level, extra_json)
    VALUES
    (:end_ts, :station, :barcode, :step_key, :score, :min_prob, :alarm_level, CAST(:extra_json AS jsonb))
    ON CONFLICT DO NOTHING
    """

    rows = []
    for _, r in df_alarm.iterrows():
        extra = {"hour_bucket": str(r.get("hour_bucket"))}
        rows.append({
            "end_ts": r["end_ts"].to_pydatetime() if isinstance(r["end_ts"], pd.Timestamp) else r["end_ts"],
            "station": str(r["station"]),
            "barcode": str(r["barcode"]),
            "step_key": str(r["step_key"]),
            "score": float(r["score"]),
            "min_prob": float(min_prob),
            "alarm_level": alarm_level,
            "extra_json": json.dumps(extra, ensure_ascii=False),
        })

    with engine.begin() as conn:
        conn.execute(text(q), rows)
    return len(rows)

✅ CELL 10) run_once (증분 감시 + sweep 테이블 + 실제 insert)

In [10]:
def run_once(engine: Engine, model: LoadedModel, cursor_key: str, window_minutes: int = 60, fetch_limit: int = 200_000):
    cur = read_cursor(engine, cursor_key)
    now = datetime.now()

    win_start = now - timedelta(minutes=window_minutes)
    start_ts = max(cur, win_start)
    end_ts = now

    df = fetch_rows(engine, model, start_ts=start_ts, end_ts=end_ts, limit=fetch_limit)
    if df.empty:
        return {"rows": 0, "cursor_old": cur, "cursor_new": cur, "inserted_alarms": 0, "sweep_df": pd.DataFrame()}

    df["end_ts"] = pd.to_datetime(df["end_ts"])
    df["station"] = df["station"].astype(str)
    df["barcode"] = df["barcode"].astype(str)
    df["step_key"] = df["step_key"].astype(str)

    scores = predict_scores(model, df)
    df_scored = df[["end_ts", "station", "barcode", "step_key"]].copy()
    df_scored["score"] = scores

    # sweep(모니터링용)
    total_hours = max(1.0, (df_scored["end_ts"].max() - df_scored["end_ts"].min()).total_seconds() / 3600.0)
    sweep_rows = []
    for mp in MIN_PROB_LIST:
        df_alarm = apply_top1_per_hour_station(df_scored, min_prob=mp)
        alarms_total = len(df_alarm)
        sweep_rows.append({
            "min_prob": mp,
            "alarms_total": alarms_total,
            "alarms_per_hour": alarms_total / total_hours,
        })
    sweep_df = pd.DataFrame(sweep_rows).sort_values("min_prob", ascending=False)

    # 실제 insert는 선택된 min_prob 1개만
    df_alarm_sel = apply_top1_per_hour_station(df_scored, min_prob=SELECTED_MIN_PROB)
    inserted = insert_alarms(engine, df_alarm_sel, min_prob=SELECTED_MIN_PROB, alarm_level="MODEL_EARLY")

    new_cursor = df["end_ts"].max().to_pydatetime()
    write_cursor(engine, cursor_key, new_cursor)

    return {
    "rows": len(df),
    "cursor_old": cur,
    "cursor_new": new_cursor,
    "inserted_alarms": inserted,
    "sweep_df": sweep_df,
    "df_alarm_sel": df_alarm_sel,   # ✅ 추가
    }

res = run_once(ENGINE, MODEL, CURSOR_KEY, window_minutes=WINDOW_MINUTES, fetch_limit=200_000)
print("[RUN_ONCE]", {k: v for k, v in res.items() if k != "sweep_df"})
display(res["sweep_df"].head(10))

[RUN_ONCE] {'rows': 0, 'cursor_old': datetime.datetime(2026, 1, 21, 5, 32, 44, 209791), 'cursor_new': datetime.datetime(2026, 1, 21, 5, 32, 44, 209791), 'inserted_alarms': 0}


""


✅ (추가) CELL 10.5) alarm_record 테이블 생성 + 이력 INSERT 함수

In [11]:
# =========================
# alarm_record (history) table
# schema: g_production_film
# table : alarm_record (없으면 생성)
# =========================

HIST_SCHEMA = "g_production_film"
HIST_TABLE  = "alarm_record"

DDL_ALARM_RECORD = f"""
CREATE TABLE IF NOT EXISTS "{HIST_SCHEMA}"."{HIST_TABLE}" (
    id          bigserial PRIMARY KEY,
    end_day     date NOT NULL,
    end_time    time NOT NULL,
    station     text NOT NULL,
    sparepart   text NOT NULL,
    type_alarm  text NOT NULL,
    created_at  timestamp NOT NULL DEFAULT now()
);

-- 중복 방지(동일 시각/스테이션/스페어파트/타입은 1번만)
DO $$
BEGIN
    IF NOT EXISTS (
        SELECT 1
        FROM pg_constraint
        WHERE conname = 'alarm_record_uniq'
    ) THEN
        ALTER TABLE "{HIST_SCHEMA}"."{HIST_TABLE}"
        ADD CONSTRAINT alarm_record_uniq UNIQUE (end_day, end_time, station, sparepart, type_alarm);
    END IF;
END $$;
"""

def ensure_alarm_record_table(engine: Engine):
    with engine.begin() as conn:
        conn.execute(text(DDL_ALARM_RECORD))
    print(f"[OK] ensured {HIST_SCHEMA}.{HIST_TABLE}")

ensure_alarm_record_table(ENGINE)

# -------------------------
# sparepart 추정 (휴리스틱)
# - step_key(=step_description/group/remark 중 선택된 것) 문자열에서 usb_a/usb_c/mini_b 탐지
# - 못 찾으면 'unknown'
# -------------------------
def infer_sparepart_from_step(step_key: str) -> str:
    s = (step_key or "").lower()
    if "usb_c" in s or "usbc" in s:
        return "usb_c"
    if "usb_a" in s or "usba" in s:
        return "usb_a"
    if "mini_b" in s or "minib" in s:
        return "mini_b"
    return "unknown"

# -------------------------
# type_alarm 분류 (운영정책에 맞게 숫자만 바꾸면 됨)
# 기본안:
# - score >= 0.80 : "긴급"
# - score >= 0.60 : "권고"
# - else          : "준비"
# -------------------------
def classify_type_alarm(score: float) -> str:
    if score >= 0.80:
        return "긴급"
    if score >= 0.60:
        return "권고"
    return "준비"

# -------------------------
# alarm_record INSERT
# df_alarm_sel: end_ts, station, step_key, score 컬럼 필요
# -------------------------
def insert_alarm_record(engine: Engine, df_alarm_sel: pd.DataFrame) -> int:
    if df_alarm_sel is None or df_alarm_sel.empty:
        return 0

    q = f"""
    INSERT INTO "{HIST_SCHEMA}"."{HIST_TABLE}"
    (end_day, end_time, station, sparepart, type_alarm)
    VALUES
    (:end_day, :end_time, :station, :sparepart, :type_alarm)
    ON CONFLICT ON CONSTRAINT alarm_record_uniq DO NOTHING
    """

    rows = []
    for _, r in df_alarm_sel.iterrows():
        ts = r["end_ts"]
        if isinstance(ts, pd.Timestamp):
            ts = ts.to_pydatetime()

        end_day  = ts.date()
        end_time = ts.time().replace(microsecond=0)

        station  = str(r["station"])
        step_key = str(r.get("step_key", ""))
        sparepart = infer_sparepart_from_step(step_key)

        # 원하면 unknown은 기록 제외도 가능 (지금은 남겨둠)
        type_alarm = classify_type_alarm(float(r["score"]))

        rows.append({
            "end_day": end_day,
            "end_time": end_time,
            "station": station,
            "sparepart": sparepart,
            "type_alarm": type_alarm,
        })

    with engine.begin() as conn:
        conn.execute(text(q), rows)

    return len(rows)

[OK] ensured g_production_film.alarm_record


✅ CELL 11) 실시간 루프(원하면 켜)

In [ ]:
while True:
    try:
        res = run_once(ENGINE, MODEL, CURSOR_KEY, window_minutes=WINDOW_MINUTES, fetch_limit=200_000)

        # ✅ rt_alarms에 넣은 "선택 알람"을 alarm_record에도 이력 저장
        df_sel = res.get("df_alarm_sel", pd.DataFrame())
        n_hist = insert_alarm_record(ENGINE, df_sel)

        print(
            "[LOOP]",
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            f"rows={res.get('rows')} inserted_rt={res.get('inserted_alarms')} inserted_hist={n_hist}",
            f"cursor_new={res.get('cursor_new')}"
        )

    except Exception as e:
        print("[ERROR]", e)

    time.sleep(POLL_SEC)

[LOOP] 2026-01-21 11:41:26 rows=0 inserted_rt=0 inserted_hist=0 cursor_new=2026-01-21 05:32:44.209791
[LOOP] 2026-01-21 11:41:37 rows=0 inserted_rt=0 inserted_hist=0 cursor_new=2026-01-21 05:32:44.209791
[LOOP] 2026-01-21 11:41:47 rows=0 inserted_rt=0 inserted_hist=0 cursor_new=2026-01-21 05:32:44.209791
[LOOP] 2026-01-21 11:41:58 rows=0 inserted_rt=0 inserted_hist=0 cursor_new=2026-01-21 05:32:44.209791
